# 26 · Inference checks: repeatability of exact inference and initialization audit (T4 GPU, about 1 h, resumable)
No training. Read-only for all runs and data. Writes `results/p2_rev4/checks/`.

**(i) Repeatability.** For the 60 circuit runs and their 60 Dense(4) partners (same fold, seed and schedule), exact inference on the
CBIS-DDSM test fold and the 301 DMID crops is run three times in this session (same checkpoint, same inputs, same batch order) and
compared with the per-image Dice stored by notebooks 14/20/22.
**(ii) Initialization audit.** For seeds 0–2 and fold 0, the global and patch-wise variants (circuit, Dense(4), trigonometric, identity)
are built with the training seed exactly as in `qseg2.run_fold`; the weights of the layers that all variants share are compared, and the
initial statistics of the code entering and leaving the 4→4 block and the gradient norms at the block are recorded on four training batches.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, gc, hashlib, random
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
FROZ_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit_frozen'
DMID_DIR = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
OUT      = f'{PROJECT}/results/p2_rev4'
ROOTS    = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val',
            'train4': f'{PROJECT}/results/p2_rev4_runs_train', 'val4': f'{PROJECT}/results/p2_rev4_runs_val'}
SCHED    = {'train': 'train', 'val': 'val', 'train4': 'train', 'val4': 'val'}
for d in [OUT] + list(ROOTS.values()): os.makedirs(d, exist_ok=True)
PROTO_FILE = f'{OUT}/PROTOCOL_rev4_prespecified.txt'
PROTO = """Paper 2 - revision 4 analyses, written before any of them was run (frozen cohort selected, crops made or runs trained).

A. Frozen evaluation on untouched patients. The pipeline is frozen: the 190 saved models, crop protocol, threshold 0.5 and metrics are
   unchanged. Cohort: CBIS-DDSM mass ROIs (same eligibility as the study: every mass ROI whose full-field image is available, no filtering)
   of patients NOT among the 208 study patients. Patients are shuffled once with Python random.Random(2026) and added whole, in that
   order, while the total stays at or below 400 ROIs (patients that would exceed 400 are skipped); selection stops at 400 ROIs.
   Crops: the study's randomized protocol (seed from the MD5 of the ROI identifier). Every saved model predicts every frozen crop; per
   crop and seed, Dice is averaged over the five fold models (as for DMID). Primary estimands: the nine model contrasts of Tables VI-VII
   under each schedule, crossed hierarchical bootstrap over patients and seeds (10,000 replicates, rng(0)), Holm within the nine
   contrasts of each schedule. Secondary: model levels, the two no-network baselines, empty-prediction rates, HD95.
B. Inference checks (no training). (i) Exact inference repeated three times within one session (same checkpoint, same inputs) and
   compared with the stored values, for the 60 circuit runs and their Dense(4) partners, on the CBIS-DDSM test fold and DMID.
   (ii) Initialization audit: for seeds 0-2 and fold 0, the global and patch-wise models (circuit, Dense(4), trigonometric, identity)
   are built with the training seed; the weights of the layers shared by all variants are compared, and the initial block input and
   output statistics and the gradient norms at the block are recorded on four training batches.
C. New training runs (training and evaluation exactly as in the study):
   C. Trigonometric control g_tr, p_tr: z = tanh(A cos((pi/2) x + phi)), A 4x4 Glorot uniform, phi uniform [0, 2 pi): 20 parameters,
       same slot as the circuit; seeds 0-2, folds 0-4, both schedules (60 runs).
   Primary estimands (CBIS-DDSM test folds): circuit - trigonometric at each placement under each schedule (4 contrasts, Holm over
   the four). Secondary: trigonometric - Dense(4) and trigonometric - identity at each placement; the same contrasts on DMID
   (exploratory) and on the frozen cohort; code-replacement drops of the trigonometric models.
   No further seeds and no further learning-rate schedule are trained.
All runs and all results are reported; none is discarded."""
if not os.path.exists(PROTO_FILE):
    open(PROTO_FILE, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
print('protocol:', open(PROTO_FILE).read().splitlines()[-1])
CODE_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{CODE_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    CODE_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (CODE_DIR, f'{PROJECT}/code'):
    if p not in sys.path: sys.path.insert(0, p)


In [ ]:
# write the revision-4 module (trigonometric block) next to qseg2.py
open(f'{PROJECT}/code/qseg2_rev4.py', 'w').write('"""Revision-4 additions to qseg2 (Paper 2): a 20-parameter trigonometric block for the 4->4 slot.\n\n    z = tanh(A cos((pi/2) x + phi)),  A in R^{4x4} (Glorot uniform),  phi in R^4 (uniform on [0, 2 pi), as the circuit angles)\n\nThe angle encoding (pi/2) x is the one the circuit uses; like the circuit outputs, the block output is a trigonometric\nfunction of the angles. Global models apply it to the 4-value code, patch-wise models to every cell with one shared set of\nweights (as the patch-wise circuit). Models: g_tr, p_tr. Call patch() once after importing qseg2.\n"""\nimport numpy as np\nimport tensorflow as tf\nfrom tensorflow.keras import layers\n\nimport qseg2\n\n\nclass TrigLayer(layers.Layer):\n    def __init__(self, n=4, **kw):\n        super().__init__(**kw)\n        self.n = n\n\n    def build(self, input_shape):\n        self.A = self.add_weight(name="A", shape=(self.n, self.n), initializer="glorot_uniform")\n        self.phi = self.add_weight(name="phi", shape=(self.n,), initializer=tf.keras.initializers.RandomUniform(0.0, 2 * np.pi))\n        super().build(input_shape)\n\n    def call(self, x):\n        c = tf.cos(np.pi / 2 * tf.cast(x, tf.float32) + self.phi)\n        return tf.tanh(tf.einsum("...i,ji->...j", c, self.A))\n\n    def compute_output_shape(self, input_shape):\n        return input_shape\n\n    def get_config(self):\n        return {**super().get_config(), "n": self.n}\n\n\n_orig_block = qseg2._block\n\n\ndef _block(z, kind, n_qubits, n_layers, patch):\n    if kind == "tr":\n        return TrigLayer(n_qubits, name="trig_replacement")(z)\n    return _orig_block(z, kind, n_qubits, n_layers, patch)\n\n\ndef patch():\n    qseg2._block = _block\n    qseg2.MODELS2.setdefault("g_tr", "Global bottleneck, trigonometric 4->4 layer (20 parameters)")\n    qseg2.MODELS2.setdefault("p_tr", "Patch-wise bottleneck, trigonometric 4->4 layer (20 parameters)")\n    return qseg2\n')


In [ ]:
import importlib, tensorflow as tf, qseg2; importlib.reload(qseg2)
import qseg2_rev4; importlib.reload(qseg2_rev4); qseg2_rev4.patch()
from qseg.config import Config
cfg = Config(work_dir=OUT); B = cfg.batch_size
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# 1 · Data and stored values
os.makedirs(f'{OUT}/checks', exist_ok=True)
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
meta = pd.read_csv(f'{DMID_DIR}/metadata.csv')
XD = np.stack([_png(f'{DMID_DIR}/images/{r}.png', False) for r in meta.roi_id]); YD = np.stack([_png(f'{DMID_DIR}/masks/{r}.png', True) for r in meta.roi_id])
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
FP = f'{PROJECT}/results/p2_full_precision'
REF = pd.concat([pd.read_csv(f'{FP}/full_{k}_{s}.csv.gz') for k in ('cbis', 'dmid_corrected') for s in ('train', 'val')], ignore_index=True)
print('DMID', XD.shape, '| CBIS', XC.shape, '| stored per-image table:', None if REF is None else REF.shape)
def dice_fast(Y, P):
    yt, yp = Y[..., 0] > 0.5, P[..., 0] >= cfg.threshold
    tp = (yt & yp).sum((1, 2)); s = yt.sum((1, 2)) + yp.sum((1, 2))
    return np.where(s == 0, 1.0, 2.0 * tp / np.maximum(s, 1))
def load(d, name):
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{d}/best.weights.h5'); return m

In [ ]:
# 2 · (i) Repeated exact inference (three passes per run, same session)
dst = f'{OUT}/checks/repeat_inference.csv'
rows = pd.read_csv(dst).to_dict('records') if os.path.exists(dst) else []
done = {(r['set'], r['model'], r['fold'], r['seed']) for r in rows}
for set_name in ('train', 'val'):
    for name in ('g_q', 'p_q', 'g_cl', 'p_cl'):
        for fold in range(5):
            for seed in range(3):
                if (set_name, name, fold, seed) in done: continue
                d = f'{ROOTS[set_name]}/{name}_fold{fold}_seed{seed}'
                m = load(d, name); _, _, te = qseg2.fold_split(DFC, fold)
                f = tf.function(lambda x: m(x, training=False))
                for ds, X, Y, ids in (('CBIS', XC[te], YC[te], DFC.roi_id.values[te]), ('DMID', XD, YD, meta.roi_id.values)):
                    P = [np.concatenate([f(tf.constant(X[i:i + B])).numpy() for i in range(0, len(X), B)]) for _ in range(3)]
                    D = [dice_fast(Y, p) for p in P]
                    r = dict(set=set_name, model=name, fold=fold, seed=seed, dataset=ds,
                             max_abs_dprob_repeat=float(max(np.abs(P[k] - P[0]).max() for k in (1, 2))),
                             max_abs_ddice_repeat=float(max(np.abs(D[k] - D[0]).max() for k in (1, 2))),
                             mean_dice=float(D[0].mean()))
                    if REF is not None:
                        s = REF[(REF.set == set_name) & (REF.model == name) & (REF.fold == fold) & (REF.seed == seed) & (REF.dataset == ds)].set_index('roi_id').dice
                        if len(s): r['max_abs_ddice_vs_stored'] = float(np.nanmax(np.abs(D[0] - s.reindex(ids).values))); r['mean_ddice_vs_stored'] = float(np.nanmean(D[0] - s.reindex(ids).values))
                    rows.append(r)
                pd.DataFrame(rows).to_csv(dst, index=False); del m; gc.collect()
                print(set_name, name, fold, seed, 'repeat max |dDice| =', rows[-1]['max_abs_ddice_repeat'], flush=True)
R = pd.DataFrame(rows); print(R.groupby('dataset')[[c for c in R.columns if c.startswith('max_')]].max().to_string())

In [ ]:
# 3 · (ii) Initialization audit (models built exactly as in qseg2.run_fold, no training)
from qseg.train import bce_dice_loss
tr, _, _ = qseg2.fold_split(DFC, 0)
VARS = ['g_q', 'g_cl', 'g_tr', 'g_id', 'p_q', 'p_cl', 'p_tr', 'p_id']
BLOCK = {'q': 'quantum_circuit', 'cl': 'classical_replacement', 'tr': 'trig_replacement'}
rows, wts = [], {}
for seed in range(3):
    for name in VARS:
        tf.keras.backend.clear_session(); tf.keras.mixed_precision.set_global_policy('float32'); qseg2.set_seeds(seed)
        m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
        wts[(seed, name)] = {w.path if hasattr(w, 'path') else w.name: w.numpy() for w in m.weights}
        pre = m.get_layer('pre_dense' if name.startswith('g_') else 'pre_conv'); sev = m.get_layer('sever')
        probe = tf.keras.Model(m.input, [pre.output, sev.input])
        blk = BLOCK.get(name[2:]); bl = m.get_layer(blk) if blk else None
        stats = []
        for b in range(4):
            xb, yb = XC[tr][b * B:(b + 1) * B], YC[tr][b * B:(b + 1) * B]
            with tf.GradientTape(persistent=True) as tape:
                cin, cout = probe(xb, training=False); tape.watch(cin)
                loss = bce_dice_loss(yb, m(xb, training=False))
            g_pre = tape.gradient(loss, pre.trainable_weights[0])
            g_blk = tape.gradient(loss, bl.trainable_weights) if bl is not None else None
            stats.append(dict(in_mean=float(np.mean(cin)), in_sd=float(np.std(cin)), out_mean=float(np.mean(cout)), out_sd=float(np.std(cout)),
                              out_sd_across_images=float(np.mean(np.std(np.reshape(cout, (cout.shape[0], -1)), 0))),
                              grad_pre_norm=float(tf.norm(g_pre)), grad_block_norm=float(np.sqrt(sum(float(tf.reduce_sum(g ** 2)) for g in g_blk))) if g_blk else np.nan))
            del tape
        rows.append(dict(seed=seed, model=name, **pd.DataFrame(stats).mean().to_dict()))
        del m; gc.collect()
S = pd.DataFrame(rows); S.to_csv(f'{OUT}/checks/init_scales.csv', index=False); print(S.round(4).to_string())
same = []
for seed in range(3):
    for place in ('g', 'p'):
        ref = wts[(seed, f'{place}_id')]
        for kind in ('q', 'cl', 'tr'):
            w = wts[(seed, f'{place}_{kind}')]
            shared = [k for k in ref if k in w and w[k].shape == ref[k].shape]
            enc = [k for k in shared if k.split('/')[0].startswith(('enc', 'bottleneck'))]
            dec = [k for k in shared if k.split('/')[0].startswith(('dec', 'post', 'output', 'pre'))]
            same.append(dict(seed=seed, placement=place, block=kind, n_shared=len(shared),
                             encoder_identical=all(np.array_equal(w[k], ref[k]) for k in enc), n_encoder=len(enc),
                             decoder_identical=all(np.array_equal(w[k], ref[k]) for k in dec), n_decoder=len(dec),
                             max_abs_diff_decoder=float(max([np.abs(w[k] - ref[k]).max() for k in dec] or [0]))))
W = pd.DataFrame(same); W.to_csv(f'{OUT}/checks/init_identity.csv', index=False); print(W.to_string())

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()